# Store/Read/Query SQLlite 

In [5]:
## Init a DataFrame
from pandas import read_csv, Series

publications = read_csv("../notebook/01-publications.csv", 
                        keep_default_na=False,
                        dtype={
                            "doi": "string",
                            "title": "string",
                            "publication year": "int",
                            "publication venue": "string",
                            "type": "string",
                            "issue": "string",
                            "volume": "string"
                        })

# Create a new column with internal identifiers for each publication
publication_internal_id = []
for idx, row in publications.iterrows():
    publication_internal_id.append("publication-" + str(idx))
    
publications.insert(0, "internalId", Series(publication_internal_id, dtype="string"))

publications


,internalId,doi,title,publication year,publication venue,type,issue,volume
0,publication-0,10.1002/cfg.304,Development of Computational Tools for the Inf...,2003,1531-6912,journal article,4,4
1,publication-1,10.1016/s1367-5931(02)00332-0,In vitro selection as a powerful tool for the ...,2002,1367-5931,journal article,3,6
2,publication-2,10.1002/9780470291092.ch20,Mechanisms of Toughening in Ceramic Matrix Com...,1981,9780470291092,book chapter,,


In [11]:
## Push a DataFrame in Sqllite
from sqlite3 import connect

with connect("publications-db.db") as con:
    publications.to_sql("Publications", con, if_exists="replace", index=False)


In [10]:
## Query SqlLite
from pandas import read_sql

with connect("publications-db.db") as con:
    query = "SELECT title FROM Publications WHERE issue==4"
    df_sql = read_sql(query, con)
    
df_sql  # show the content of the result of the query

,title
0,Development of Computational Tools for the Inf...


# Store/Read/Query Blazegraph

In [12]:
# Init a Graph

from rdflib import URIRef, Literal, Graph, RDF

a_string = Literal("a string with this value")
a_number = Literal(42)
a_boolean = Literal(True)

# classes of resources
JournalArticle = URIRef("https://schema.org/ScholarlyArticle")
BookChapter = URIRef("https://schema.org/Chapter")
Journal = URIRef("https://schema.org/Periodical")
Book = URIRef("https://schema.org/Book")

# attributes related to classes
doi = URIRef("https://schema.org/identifier")
publicationYear = URIRef("https://schema.org/datePublished")
title = URIRef("https://schema.org/name")
issue = URIRef("https://schema.org/issueNumber")
volume = URIRef("https://schema.org/volumeNumber")
identifier = URIRef("https://schema.org/identifier")
name = URIRef("https://schema.org/name")

# relations among classes
publicationVenue = URIRef("https://schema.org/isPartOf")

In [14]:
# Insert into a Graph

my_graph = Graph()
base_url = "https://thinkcompute.github.io/res/"

publications = read_csv("../notebook/01-publications.csv", 
                        keep_default_na=False,
                        dtype={
                            "doi": "string",
                            "title": "string",
                            "publication year": "int",
                            "publication venue": "string",
                            "type": "string",
                            "issue": "string",
                            "volume": "string"
                        })

for idx, row in publications.iterrows():
    
    local_id = "publication-" + str(idx)
    
    # The shape of the new resources that are publications is
    # 'https://comp-data.github.io/res/publication-<integer>'
    subj = URIRef(base_url + local_id)
    
    if row["type"] == "journal article":
        my_graph.add((subj, RDF.type, JournalArticle))

        # These two statements applies only to journal articles
        my_graph.add((subj, issue, Literal(row["issue"])))
        my_graph.add((subj, volume, Literal(row["volume"])))
    else:
        my_graph.add((subj, RDF.type, BookChapter))
    
    my_graph.add((subj, name, Literal(row["title"])))
    my_graph.add((subj, identifier, Literal(row["doi"])))
    
    # The original value here has been casted to string since the Date type
    # in schema.org ('https://schema.org/Date') is actually a string-like value
    my_graph.add((subj, publicationYear, Literal(str(row["publication year"]))))
    

In [15]:
# Store in Blazegraph

from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

store = SPARQLUpdateStore()

# The URL of the SPARQL endpoint is the same URL of the Blazegraph
# instance + '/sparql'
endpoint = 'http://127.0.0.1:9999/blazegraph/sparql'

# It opens the connection with the SPARQL endpoint instance
store.open((endpoint, endpoint))

for triple in my_graph.triples((None, None, None)):
   store.add(triple)
    
# Once finished, remeber to close the connection
store.close()

In [18]:
# Query Blazegraph from python

# Install "sparql_dataframe" first
# pip install sparql_dataframe
# -----

from sparql_dataframe import get

endpoint = "http://127.0.0.1:9999/blazegraph/sparql"
query = """
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX schema: <https://schema.org/>

SELECT *
WHERE {
    ?pub rdf:type schema:ScholarlyArticle .
    ?pub schema:name ?title .
}
"""
df_sparql = get(endpoint, query, True)
df_sparql

,pub,title
0,https://thinkcompute.github.io/res/publication-1,In vitro selection as a powerful tool for the ...
1,https://thinkcompute.github.io/res/publication-0,Development of Computational Tools for the Inf...
